# CADAC ROCKET6G — training data generation

Generates `(x, p, xdot)` samples from CADAC's three-stage launch vehicle for
gray-box state-space identification.

**Pipeline:** fetch → patch → build → Monte Carlo → parse → validate → `.npz` on Drive.

### Start at 50 runs

At 12 runs the model was data-limited (train 0.500 / val 0.521 — a small gap, so
capacity was not the constraint). That does not say whether the curve flattens at
50 or at 200. Generate 50, train, read the validation curve, and extend only if it
is still improving.

Extending costs nothing: raise `N_RUNS` and re-run. Chunk `i` always uses
`seed + i*1000` and `run_id` offset `i*chunk_size` regardless of the total, so
existing chunks are skipped and only new ones compute.

### Measured cost

~17 s/run on a Raspberry Pi 5 at `plot_step=0.01`; Colab is typically 2–3× faster.
Each run writes ~63 MB of ASCII, deleted after parsing, and adds ~3.3 MB to the
compressed `.npz`.

| | 50 runs | 200 runs |
|---|---|---|
| wall clock (Colab est.) | ~6–15 min | ~25–60 min |
| peak disk | ~630 MB | ~630 MB |
| final `.npz` | ~165 MB | ~660 MB |

Chunk 1 is timed and the total projected before the rest run, so a bad estimate
surfaces about a minute in.

## 1. Environment

In [ ]:
!g++ --version | head -1
!nproc --all | xargs echo 'CPUs:'
!df -h /content | tail -1 | awk '{print "disk free: "$4}'
import sys; print('python:', sys.version.split()[0])

## 2. Get the framework code

Clones the repo if it is populated; otherwise prompts for an upload.

The CADAC simulation sources are committed under `cadac/`, so the clone brings
them too and nothing else has to be fetched — no second `git clone` of a
third-party repository per session. If you fall back to uploading, take
`generator.py` plus `physics.py`; without `cadac/` the generator clones CADAC
upstream instead, which still works but costs a download each runtime.

In [ ]:
import os, sys, shutil, subprocess, pathlib

REPO = 'https://github.com/alican30alicanexe-alt/systemid'
CODE = pathlib.Path('/content/systemid')

# Always refresh; never trust an existing checkout.
#
# This cell used to skip the clone whenever CODE/*.py already existed. A runtime
# that had cloned earlier -- before a push, or in a previous session -- therefore
# kept running old code while the notebook itself was new. That produced a full
# 50-run dataset against the pre-FSPB schema before anything complained. The clone
# is seconds; the generation it protects is hours.
if (CODE / '.git').is_dir():
    subprocess.run(['git', '-C', str(CODE), 'fetch', '--depth', '1', 'origin', 'main'],
                   capture_output=True, text=True)
    subprocess.run(['git', '-C', str(CODE), 'reset', '--hard', 'FETCH_HEAD'],
                   capture_output=True, text=True)
else:
    shutil.rmtree(CODE, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(CODE)],
                   capture_output=True, text=True)

if not (CODE / 'generator.py').exists():
    print('clone did not yield generator.py; upload the framework .py files instead')
    from google.colab import files
    CODE.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (CODE / name).write_bytes(data)

sys.path.insert(0, str(CODE))

# Drop anything already imported, or a re-run of this cell keeps the stale module
# objects and the refresh above achieves nothing.
for _m in [m for m in list(sys.modules) if m.split('.')[0] in {
        'generator', 'physics', 'dataset', 'model', 'trainer', 'evaluate',
        'identifiability'}]:
    del sys.modules[_m]

head = subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--format=%h %s'],
                      capture_output=True, text=True).stdout.strip()
print('code at', head or '(no git metadata)')

import generator

# Fail here, in seconds, rather than after a generation run. Everything downstream
# assumes this schema; a mismatch means the refresh above did not take.
missing = {'etax', 'zetx'} - set(generator.DEFAULT_PARAMS)
if missing or not hasattr(generator, 'TRUTH_VARS'):
    raise RuntimeError(
        f'stale generator.py: missing params {sorted(missing)}'
        f'{" and TRUTH_VARS" if not hasattr(generator, "TRUTH_VARS") else ""}. '
        'The clone did not refresh -- confirm the fix is pushed to origin/main, '
        'then Runtime > Disconnect and delete runtime and start again.'
    )

print('states:', generator.DEFAULT_STATE)
print('params:', generator.DEFAULT_PARAMS)
print('truth :', generator.TRUTH_VARS, '(stored outside p)')

vendored = generator.VENDORED_CADAC / 'ROCKET6G' / 'newton.cpp'
print('\nCADAC source:', 'vendored (no download)' if vendored.is_file()
      else f'will clone {generator.CADAC_REPO}')


## 3. Mount Drive

Chunks and the merged dataset live on Drive, so a Colab disconnect costs at most
the chunk in flight.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = pathlib.Path('/content/drive/MyDrive/systemid/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('output ->', DATA_DIR)

## 4. Configuration

`int_step` is CADAC's integration step and drives almost all of the runtime.
Leave it at 0.001 — raising it is the only real speed lever but it changes the
fidelity of the data being identified against.

Dispersions are applied by CADAC's own `MONTE` block as `GAUSS mean sigma` on
`input_insertion.asc` scalars.

In [ ]:
# @title Campaign settings { run: "auto" }
N_RUNS     = 50      # @param {type:"integer"}
CHUNK_SIZE = 10      # @param {type:"integer"}
PLOT_STEP  = 0.01    # @param {type:"number"}
INT_STEP   = 0.001   # @param {type:"number"}
ENDTIME    = 190.0   # @param {type:"number"}
SEED       = 1234    # @param {type:"integer"}

# @markdown Dispersions — CADAC applies these as `GAUSS mean sigma`
THTBDX_MEAN  = 90.0      # @param {type:"number"}
THTBDX_SIGMA = 2.0       # @param {type:"number"}
PSIBDX_MEAN  = -83.0     # @param {type:"number"}
PSIBDX_SIGMA = 2.0       # @param {type:"number"}
VMASS0_MEAN  = 48984.0   # @param {type:"number"}
VMASS0_SIGMA = 500.0     # @param {type:"number"}
SPI_MEAN     = 279.2     # @param {type:"number"}
SPI_SIGMA    = 3.0       # @param {type:"number"}

OUT = DATA_DIR / f'rocket6g_{N_RUNS}.npz'

cfg = generator.GeneratorConfig(
    work_dir=pathlib.Path('/content/cadac_work'),
    out_path=OUT,
    n_runs=N_RUNS, seed=SEED,
    plot_step=PLOT_STEP, int_step=INT_STEP, endtime=ENDTIME,
    dispersions={
        'thtbdx': (THTBDX_MEAN, THTBDX_SIGMA),
        'psibdx': (PSIBDX_MEAN, PSIBDX_SIGMA),
        'vmass0': (VMASS0_MEAN, VMASS0_SIGMA),
        'spi':    (SPI_MEAN,    SPI_SIGMA),
    },
)

# Measured: 63 MB of ASCII per run at plot_step=0.01, scaling inversely with it.
mb_per_run = 63.0 * (0.01 / PLOT_STEP)
n_chunks = -(-N_RUNS // CHUNK_SIZE)
print(f'{N_RUNS} runs in {n_chunks} chunk(s) of <= {CHUNK_SIZE}  ->  {OUT}')
print(f'peak disk ~{mb_per_run * min(CHUNK_SIZE, N_RUNS):.0f} MB '
      f'(one chunk; deleted after parsing, never {mb_per_run * N_RUNS / 1e3:.1f} GB at once)')

## 5. Fetch, patch and build CADAC

Stock CADAC needs three source patches before its output is usable. Expect all
three in the log below:

1. **Plot flags** — the stock plot file carries position/velocity only in polar
   form. `SBII`/`VBII`/`ABII`/`FSPB`/`WBIB`/`FAPB` are in the module array but not
   flagged for output. Adding `plot` takes the file from 108 to 126 columns.
   `ABII` is CADAC's own acceleration, so it doubles as ground truth.
2. **Precision 6 → 14 digits** — `Hyper::plot_data` never sets stream precision.
   `|SBII|` is ~6.37e6 m, which at 6 significant digits quantises to ±5 m;
   differenced over a plot step that is ±100 m/s of pure round-off.
3. **Column width 16 → 26** — at 14 digits the numbers overflow the field and run
   together with no separator. The two are a pair; neither works alone.

This compiles once. Only `input.asc` changes between chunks.

In [ ]:
import time
t0 = time.monotonic()
generator.prepare(cfg)
print(f'\nbuild complete in {time.monotonic() - t0:.0f}s')

## 6. Generate

Each chunk is simulated, parsed to `chunks/chunk_XXX.npz` on Drive, and its
`plot1.asc` deleted before the next starts. Re-running skips chunks that already
exist — that is both the resume path after a disconnect and the extend path when
`N_RUNS` goes up.

⚠️ **Delete `chunks/` if it predates the `fspb` column or the `etax`/`zetx`
parameters.** Chunks are cached by index alone, so a schema change does not
invalidate them. `merge_datasets` now raises on a chunk missing an array it
expects, and the metadata check catches a stale `param_names`, but the cheap fix
is to delete the directory whenever the config changed.

The per-chunk `FD vs CADAC ABII` line is the first validation: it compares our
finite differences against CADAC's own computed acceleration. Measured median
2.3e-06 m/s² against a ~41 m/s² typical acceleration. If it is orders of
magnitude worse, the precision patch did not take.

In [ ]:
t0 = time.monotonic()
out = generator.generate_chunked(cfg, chunk_size=CHUNK_SIZE, chunk_dir=DATA_DIR / 'chunks')
print(f'\ntotal {(time.monotonic() - t0) / 60:.1f} min -> {out}')

## 7. Validate

Two gates, in increasing strength.

**Residual report.** With aerodynamics the only unknown, the residual left by
analytical physics must be near zero in vacuum and grow with dynamic pressure.
A residual flat in `pdynmc` means a physics module is wrong, and the network would
bury that error inside the learned correction as if it were aerodynamics.

**FSPB comparison.** `newton.cpp` computes `ABII = ~TBI*FSPB + ~TGI*GRAVG`, so with
gravity and thrust modelled correctly the residual must equal
`~TBI * (FSPB - FPB/vmass)` *exactly* — not merely correlate with `pdynmc`. This is
strictly stronger, and it is the only check that catches an error which happens to
grow with dynamic pressure. The TVC bug did exactly that: it was live only during
first-stage boost, where `pdynmc` peaks, so it sailed through the residual report
and was caught here.

Expected: residual report monotonic and reaching ~5.2 m/s² at max-Q; FSPB
disagreement median ~4e-05 m/s², ~8e-06 at max-Q.

In [ ]:
import numpy as np

d = np.load(out, allow_pickle=True)
print('samples:', len(d['x']), ' runs:', len(np.unique(d['run_id'])))
print('states:', list(d['state_names']))
print('params:', list(d['param_names']))
print('dtype :', d['x'].dtype, '(float64 -- do not store positions as float32)')

# FSPB is ground truth, deliberately outside `p` so the network cannot read it.
if 'fspb' not in d:
    raise RuntimeError('no fspb array -- this dataset predates the truth column; '
                       'delete chunks/ and regenerate')
print('truth :', list(d['fspb_names']), '(stored outside p -- not a network input)')

# d(SBII)/dt must equal VBII by definition; deviation is finite-difference truncation.
kin = np.abs(d['xdot'][:, :3] - d['x'][:, 3:6]).max()
print(f'\nkinematic identity max|d(SBII)/dt - VBII| = {kin:.3e} m/s')

try:
    import torch
    from physics import StateLayout, residual_report, aerodynamic_truth

    print()
    residual = residual_report(str(out))

    layout = StateLayout(list(d['state_names']), list(d['param_names']))
    x, p, t = (torch.tensor(d[k]) for k in ('x', 'p', 't'))
    fspb = torch.tensor(d['fspb'])
    vel = layout.s_slice('VBII')

    # Shared with PropulsionModule, so the two cannot drift: any difference here
    # would otherwise read as a physics error that is really just two
    # transcriptions of tvc.cpp disagreeing.
    truth = aerodynamic_truth(p, fspb, layout, t)
    err = (residual[:, vel] - truth).norm(dim=1)

    print('\n  FSPB comparison (residual vs CADAC ground truth)')
    print(f'    |diff| median {err.median():.3e}  p99 {err.quantile(0.99):.3e} m/s^2')
    q = p[:, layout.p('pdynmc')]
    maxq = q >= 1e4
    if maxq.sum() > 10:
        print(f'    at max-Q: |truth| {truth[maxq].norm(dim=1).median():.4f}  '
              f'|diff| {err[maxq].median():.3e} m/s^2')
        ok = float(err[maxq].median()) < 1e-3
        print(f'    {"OK - analytical physics matches CADAC" if ok else "FAIL - a module disagrees with CADAC; do not train"}')
except ImportError as exc:
    print(f'\n[skip] physics gate unavailable ({exc}); run it before training')

## 8. Trajectory plots

Confirms the dispersions produced genuinely different launches rather than
50 copies of the same ascent.

In [ ]:
import matplotlib.pyplot as plt

names = list(d['param_names'])
run_id, t, x, p = d['run_id'], d['t'], d['x'], d['p']
runs = np.unique(run_id)[:8]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for r in runs:
    m = run_id == r
    axes[0].plot(t[m], p[m][:, names.index('alt')] / 1e3, lw=1)
    axes[1].plot(t[m], np.linalg.norm(x[m][:, 3:6], axis=1), lw=1)
    axes[2].plot(t[m], p[m][:, names.index('vmass')], lw=1)

for ax, label in zip(axes, ['altitude (km)', 'inertial speed (m/s)', 'vehicle mass (kg)']):
    ax.set_xlabel('time since launch (s)')
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
fig.suptitle(f'{len(runs)} dispersed ROCKET6G ascents')
fig.tight_layout()
plt.show()

# Spread across the whole campaign, not just the plotted subset.
first = np.array([np.flatnonzero(run_id == r)[0] for r in np.unique(run_id)])
for key in ('thtbdx', 'psibdx', 'vmass'):
    v = p[first][:, names.index(key)]
    print(f'{key:8s} initial: mean={v.mean():10.3f}  std={v.std():8.3f}  '
          f'range=[{v.min():.3f}, {v.max():.3f}]')

## Next

**To extend the campaign:** raise `N_RUNS` in step 4 and re-run steps 4–8. Existing
chunks are skipped, only new ones compute, and `run_id`s do not collide. Note the
output filename carries the run count, so `rocket6g_50.npz` and `rocket6g_200.npz`
coexist.

**To train:** the dataset is ready for `dataset.build_loaders(...)`. Splitting is by
trajectory, never by sample — consecutive samples are one plot step apart and are
near-duplicates, so a per-sample split reports a flattering validation loss
regardless of real generalisation.

```python
from dataset import build_loaders
train_loader, val_loader, test, meta = build_loaders(OUT, batch_size=2048, seed=0)
```